In [1]:
import pandas as pd
df = pd.read_csv("/kaggle/input/phincparallel-hinglish-corpus-machine-translation/English-Hindi code-mixed parallel corpus.csv")
df.head()

,Sentence,English_Translation
0,@someUSER congratulations on you celebrating b...,@some users congratulate you for celebrating B...
1,@LoKarDi_RT uske liye toh bahot kuch karna pad...,"@Lokardi_ rat we should a lot more for that, b..."
2,@slimswamy yehi to hum semjhane ki koshish kar...,"@Slimswami ehi, this is what i'm expecting you..."
3,@DramebaazKudi cake kaha hai ??,@Where is Dramebajakudi where is the cake?
4,@someUSER i'm in hawaii at the moment . home ...,@some user Don't want to come home next friday...


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# Load dataset
df = pd.read_csv("/kaggle/input/phincparallel-hinglish-corpus-machine-translation/English-Hindi code-mixed parallel corpus.csv")

# Rename columns if required
df.columns = ["Sentence", "English_Translation"]

# Remove NaN values
df.dropna(inplace=True)

# Sample output
print(df.head())


                                            Sentence  \
0  @someUSER congratulations on you celebrating b...   
1  @LoKarDi_RT uske liye toh bahot kuch karna pad...   
2  @slimswamy yehi to hum semjhane ki koshish kar...   
3                    @DramebaazKudi cake kaha hai ??   
4  @someUSER i'm in hawaii at the moment .  home ...   

                                 English_Translation  
0  @some users congratulate you for celebrating B...  
1  @Lokardi_ rat we should a lot more for that, b...  
2  @Slimswami ehi, this is what i'm expecting you...  
3         @Where is Dramebajakudi where is the cake?  
4  @some user Don't want to come home next friday...  


In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import nltk

nltk.download('punkt')  # Ensure NLTK tokenizer is available


df.dropna(inplace=True)  # Remove missing values

# Tokenization
def tokenize(sentence):
    return nltk.word_tokenize(sentence.lower())

# Apply tokenization
df["hinglish_tokens"] = df["Sentence"].apply(tokenize)
df["english_tokens"] = df["English_Translation"].apply(tokenize)

# Build vocabulary
hinglish_counter = Counter([word for tokens in df["hinglish_tokens"] for word in tokens])
english_counter = Counter([word for tokens in df["english_tokens"] for word in tokens])

hinglish_vocab = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"] + [word for word, _ in hinglish_counter.most_common()]
english_vocab = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"] + [word for word, _ in english_counter.most_common()]

# Word-to-index mappings
hinglish_word2idx = {word: idx for idx, word in enumerate(hinglish_vocab)}
english_word2idx = {word: idx for idx, word in enumerate(english_vocab)}

hinglish_idx2word = {idx: word for word, idx in hinglish_word2idx.items()}
english_idx2word = {idx: word for word, idx in english_word2idx.items()}

print("Hinglish vocab size:", len(hinglish_vocab))
print("English vocab size:", len(english_vocab))


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Hinglish vocab size: 27231
English vocab size: 20503


In [4]:
class HinglishDataset(Dataset):
    def __init__(self, df, hinglish_word2idx, english_word2idx, max_len=50):
        self.hinglish_sentences = df["hinglish_tokens"].tolist()
        self.english_sentences = df["english_tokens"].tolist()
        self.hinglish_word2idx = hinglish_word2idx
        self.english_word2idx = english_word2idx
        self.max_len = max_len  # Max sentence length for padding

    def __len__(self):
        return len(self.hinglish_sentences)

    def sentence_to_idx(self, sentence, word2idx):
        indexed = [word2idx.get(word, word2idx["<UNK>"]) for word in sentence]
        return [word2idx["<SOS>"]] + indexed[:self.max_len] + [word2idx["<EOS>"]]

    def pad_sequence(self, sequence):
        return sequence + [self.hinglish_word2idx["<PAD>"]] * (self.max_len + 2 - len(sequence))

    def __getitem__(self, idx):
        hinglish_seq = self.sentence_to_idx(self.hinglish_sentences[idx], self.hinglish_word2idx)
        english_seq = self.sentence_to_idx(self.english_sentences[idx], self.english_word2idx)

        hinglish_seq = self.pad_sequence(hinglish_seq)
        english_seq = self.pad_sequence(english_seq)

        return torch.tensor(hinglish_seq), torch.tensor(english_seq)

# Create dataset
dataset = HinglishDataset(df, hinglish_word2idx, english_word2idx)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Check sample
hinglish_sample, english_sample = next(iter(dataloader))
print("Sample Hinglish (Indices):", hinglish_sample[0])
print("Sample English (Indices):", english_sample[0])


Sample Hinglish (Indices): tensor([    2,   118,   290,   199,    19, 24914,     3,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])
Sample English (Indices): tensor([    2,    20,    41,   323, 18894, 18895,     3,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])


In [5]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers=num_layers,
                            bidirectional=True, batch_first=True)

    def forward(self, input_seq):
        embedded = self.embedding(input_seq)  # [batch, seq_len, embed_size]
        outputs, hidden = self.lstm(embedded)  # outputs: [batch, seq_len, hidden_size*2]
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 3, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size, seq_len, _ = encoder_outputs.shape
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)  # Repeat hidden state for each encoder output
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # The LSTM will receive the embedding concatenated with the context vector.
        self.lstm = nn.LSTM(embed_size + hidden_size * 2, hidden_size, num_layers=num_layers, batch_first=True)
        self.attention = Attention(hidden_size)
        self.out = nn.Linear(hidden_size * 3, vocab_size)

    def forward(self, input_token, hidden, encoder_outputs):
        # Ensure input_token is 2D (batch, 1)
        if input_token.dim() == 1:
            input_token = input_token.unsqueeze(1)
        # Now embedded: (batch, 1, embed_size)
        embedded = self.embedding(input_token)
        
        # Get last layer's hidden state from decoder hidden (shape: (batch, hidden_size))
        h_dec = hidden[0][-1]
        
        # Compute attention weights using encoder_outputs (batch, seq_len, hidden_size*2)
        attn_weights = self.attention(h_dec, encoder_outputs)  # (batch, seq_len)
        
        # Compute context vector with attention weights
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)  # (batch, 1, hidden_size*2)
        
        # Concatenate the embedded token and context vector along the last dimension
        lstm_input = torch.cat((embedded, context), dim=2)  # (batch, 1, embed_size + hidden_size*2)
        
        # Pass through LSTM
        output, hidden = self.lstm(lstm_input, hidden)  # output: (batch, 1, hidden_size)
        
        # Remove the time dimension (squeeze dimension 1)
        output = output.squeeze(1)    # (batch, hidden_size)
        context = context.squeeze(1)  # (batch, hidden_size*2)
        
        # Concatenate output and context to feed into the final layer
        final_output = self.out(torch.cat((output, context), dim=1))  # (batch, vocab_size)
        
        return final_output, hidden, attn_weights



class PointerGenerator(nn.Module):
    def __init__(self, hinglish_vocab_size, english_vocab_size, embed_size, hidden_size):
        super(PointerGenerator, self).__init__()
        self.encoder = Encoder(hinglish_vocab_size, embed_size, hidden_size)
        self.decoder = Decoder(english_vocab_size, embed_size, hidden_size)

    def forward(self, hinglish_seq, english_seq):
        encoder_outputs, encoder_hidden = self.encoder(hinglish_seq)
        decoder_hidden = (encoder_hidden[0][:1], encoder_hidden[1][:1])  # Only take last layer
        input_token = english_seq[:, 0]  # Start with <SOS>
        outputs = []
        for t in range(1, english_seq.shape[1]):
            output, decoder_hidden, _ = self.decoder(input_token, decoder_hidden, encoder_outputs)
            outputs.append(output.unsqueeze(1))
            input_token = english_seq[:, t]
        return torch.cat(outputs, dim=1)


In [6]:
import torch.optim as optim

# Model Hyperparameters
EMBED_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 1
HINGLISH_VOCAB_SIZE = len(hinglish_vocab)
ENGLISH_VOCAB_SIZE = len(english_vocab)

# Instantiate the model
model = PointerGenerator(HINGLISH_VOCAB_SIZE, ENGLISH_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Loss Function (CrossEntropyLoss) & Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=hinglish_word2idx["<PAD>"])  # Ignore padding
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [7]:
NUM_EPOCHS = 10  # Increase for better training
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Move model to GPU if available
model.to(DEVICE)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (hinglish_batch, english_batch) in enumerate(dataloader):
        hinglish_batch, english_batch = hinglish_batch.to(DEVICE), english_batch.to(DEVICE)

    # Reset gradients
        optimizer.zero_grad()

    # Forward pass
        output = model(hinglish_batch, english_batch)

    # Ensure correct output shape
        output = output.contiguous().view(-1, ENGLISH_VOCAB_SIZE)  # [batch * seq_len, vocab_size]
        target = english_batch[:, 1:].contiguous().view(-1)  # [batch * seq_len]

    # Compute loss
        loss = criterion(output, target)

    # Backward pass
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        
        # Print progress
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{batch_idx}/{len(dataloader)}], Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} Completed - Average Loss: {epoch_loss/len(dataloader):.4f}")

# Save the trained model
torch.save(model.state_dict(), "pointer_generator_model.pth")
print("Model saved successfully!")


Epoch [1/10], Step [0/430], Loss: 9.9270
Epoch [1/10], Step [100/430], Loss: 6.4063
Epoch [1/10], Step [200/430], Loss: 6.2874
Epoch [1/10], Step [300/430], Loss: 5.9067
Epoch [1/10], Step [400/430], Loss: 5.5103
Epoch 1 Completed - Average Loss: 6.1666
Epoch [2/10], Step [0/430], Loss: 4.9394
Epoch [2/10], Step [100/430], Loss: 4.9508
Epoch [2/10], Step [200/430], Loss: 5.0487
Epoch [2/10], Step [300/430], Loss: 5.0405
Epoch [2/10], Step [400/430], Loss: 4.5527
Epoch 2 Completed - Average Loss: 4.9434
Epoch [3/10], Step [0/430], Loss: 4.0305
Epoch [3/10], Step [100/430], Loss: 3.8194
Epoch [3/10], Step [200/430], Loss: 3.7000
Epoch [3/10], Step [300/430], Loss: 3.9692
Epoch [3/10], Step [400/430], Loss: 3.8685
Epoch 3 Completed - Average Loss: 3.9227
Epoch [4/10], Step [0/430], Loss: 3.1178
Epoch [4/10], Step [100/430], Loss: 3.0621
Epoch [4/10], Step [200/430], Loss: 3.2543
Epoch [4/10], Step [300/430], Loss: 3.2071
Epoch [4/10], Step [400/430], Loss: 3.0259
Epoch 4 Completed - Avera

In [8]:
import torch

def generate_code_switched_sentence(model, sentence, max_len=50):
    model.eval()

    # Tokenize & Convert to Indices
    tokens = tokenize(sentence)  # Assuming tokenize function exists
    input_seq = [hinglish_word2idx.get(word, hinglish_word2idx["<UNK>"]) for word in tokens]
    input_seq = torch.tensor([input_seq], device=DEVICE)  # Add batch dimension

    with torch.no_grad():
        # Encode input sequence
        encoder_outputs, encoder_hidden = model.encoder(input_seq)
        decoder_hidden = (encoder_hidden[0][:1], encoder_hidden[1][:1])  # Take only last layer

        input_token = torch.tensor([[english_word2idx["<SOS>"]]], device=DEVICE)
        generated_sentence = []

        for _ in range(max_len):
            # Forward pass through decoder
            output, decoder_hidden, _ = model.decoder(input_token, decoder_hidden, encoder_outputs)

            # ✅ Ensure correct shape: (batch_size, vocab_size)
            output = output.squeeze(1)  # Remove seq_len dimension if present
            print("Decoder Output Shape:", output.shape)  # Should be (1, vocab_size)
            # ✅ Fix shape mismatch in argmax
            predicted_idx = output.argmax(dim=-1).item()  # Extract word index

            if predicted_idx == english_word2idx["<EOS>"]:
                break  # Stop at end-of-sequence token

            generated_sentence.append(english_idx2word.get(predicted_idx, "<UNK>"))

            # ✅ Ensure correct tensor shape for next input
            input_token = torch.tensor([[predicted_idx]], device=DEVICE, dtype=torch.long)

    return " ".join(generated_sentence)

# Test the function
test_sentence = "mera naam Rahul hai aur mai engineer hoon"
print("Generated:", generate_code_switched_sentence(model, test_sentence))


Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Generated: aur name and in the name and i 'm taking the song and i am having years


In [9]:
# Load model checkpoint
model_path = "/kaggle/working/pointer_generator_model.pth"
model = PointerGenerator(HINGLISH_VOCAB_SIZE, ENGLISH_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)

# Load the trained model weights
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()  # Set to evaluation mode

# Test sentence
test_sentence = "my name is ayusj "
print("Generated:", generate_code_switched_sentence(model, test_sentence))


Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Decoder Output Shape: torch.Size([1, 20503])
Generated: @ alllahdin birthday is my constitution .


<ipython-input-9-1171d23b5013>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=DEVICE))
